In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("BRONZE LAYER - AUTO LOADER BATCH PROCESSING")
print("=" * 80)

# Create database
spark.sql("""
    CREATE DATABASE IF NOT EXISTS dm2_credit_risk
    COMMENT 'Data Management 2 - Credit Risk Final Project'
""")
print("Database: dm2_credit_risk")

# Drop existing table to avoid schema mismatch
spark.sql("DROP TABLE IF EXISTS dm2_credit_risk.bronze_loan_apps")

# Auto Loader - reads new CSV files as they arrive
df_bronze_csv = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/data_management/checkpoints/schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/workspace/default/data_management/batches/")
)

# Write stream to Delta table
(df_bronze_csv.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/workspace/default/data_management/checkpoints/bronze")
    .option("mergeSchema", "true")
    .trigger(once=True)
    .toTable("dm2_credit_risk.bronze_loan_apps")
)

print(f"Bronze CSV batch processed via Auto Loader")

In [0]:
import requests

# Fetch live World Bank GDP data
countries = ["DEU", "USA", "GBR", "FRA", "IND", "BRA", "CAN", "JPN", "CHN", "AUS"]
all_data = []

for country in countries:
    url = f"https://api.worldbank.org/v2/country/{country}/indicator/NY.GDP.MKTP.CD?format=json&per_page=10"
    response = requests.get(url)
    records = response.json()[1]
    if records:
        for r in records:
            all_data.append({
                "country": r["countryiso3code"],
                "country_name": r["country"]["value"],
                "year": r["date"],
                "gdp": float(r["value"]) if r["value"] is not None else None
            })

df_bronze_json = spark.createDataFrame(all_data)
df_bronze_json.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dm2_credit_risk.bronze_world_bank")
print(f"JSON: {df_bronze_json.count():,} rows loaded")
print("\nBRONZE LAYER COMPLETE")